# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

3. **tsfresh** - Time-series feature extraction & exploration

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [0]:
CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"

# TABLE_NAME = "credit_card_transactions"
TABLE_NAME = "hdfc_demo_bank_transactions"

FEATURE_SCHEMA_NAME = "feature_store"
# FEATURE_TABLE_NAME = "hdfc_demo_credit_card_trans_tsfresh"
FEATURE_TABLE_NAME = "hdfc_demo_cust_trans_tsfresh"


# Table Names
tsfresh_feature_table_name = f"{CATALOG_NAME}.{FEATURE_SCHEMA_NAME}.{FEATURE_TABLE_NAME}"

credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

# Reading Original Data
df = spark.read.table(credit_card_transactions_table_name)

print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
df.printSchema()

In [0]:
display(df.limit(5))

In [0]:
display(df.filter(df.amount.isNull()).limit(10))

---
## 6. TSFresh - Time-Series Feature Extraction

Extract time-series features per entity (e.g., per credit card) using tsfresh.
These are entity-level aggregates (one row per `cc_num`) that get joined back
to the transaction-level DataFrame so each transaction inherits its entity's
time-series statistics.

Supports three extraction modes:
- **minimal** (~10 features per column)
- **efficient** (~100 features per column)
- **comprehensive** (~750 features per column)

In [0]:
!pip install tsfresh

In [0]:
# For Bank Transactions
# index_col = "transaction_id"
index_col = "customer_id"
target_entities=["amount"]
time_index="transaction_date"

In [0]:
from backend.core.features.tsfresh_engine import TSFreshEngine, FeatureExtractionMode


ts_engine = TSFreshEngine(
    spark=spark,
    max_rows_for_pandas=None,
    n_jobs=0,  # use all cores
    verbose=True
)

# Filter out rows with null amounts before feature extraction
df_clean = df.filter(df.amount.isNotNull())

# Extract time-series features in minimal mode for speed
ts_result = ts_engine.extract_features(
    spark_df=df_clean,
    # id_column="trans_num",                     # entity: credit card number
    id_column=index_col,
    time_column=time_index,     # temporal column
    value_columns=target_entities,                   # numeric columns to extract from
    mode=FeatureExtractionMode.MINIMAL,
    # mode=FeatureExtractionMode.EFFICIENT,
    impute_missing=True
)

# List all columns to drop from the main DataFrame
_cols_meta = [index_col, time_index]
col_to_drop = df.columns
# print("Columns to Drop: ", col_to_drop, type(col_to_drop))
for _col in _cols_meta:
    # print(_col)
    col_to_drop.remove(_col)
    
print(f"\nTSFRESH RESULTS:")
print(f"  Features extracted: {len(ts_result.feature_names)}")
print(f"  Extraction time: {ts_result.extraction_time:.2f}s")
print(f"  Extraction mode: {ts_result.extraction_mode.value}")
if ts_result.warnings:
    print(f"  Warnings: {ts_result.warnings}")

In [0]:
# Preview extracted time-series features
print(f"TSFresh feature names ({len(ts_result.feature_names)}):")
for name in ts_result.feature_names[:15]:
    print(f"  - {name}")

ts_result.feature_matrix.head()

In [0]:
ts_result.feature_matrix.shape

In [0]:
# Filter to only statistically relevant features
import pandas as pd
from backend.core.utils.spark_pandas_bridge import spark_to_pandas_safe, pandas_to_spark

# Build per-entity target: majority label per cc_num
# target_per_entity = (
#     df.groupBy("trans_num")
#     .agg({"is_fraud": "max"})
#     .toPandas()
#     .set_index("trans_num")["max(is_fraud)"]
# )

target_per_entity = (
    df.groupBy(index_col)
    .agg({"amount": "sum"})
    .toPandas()
    .set_index(index_col)["sum(amount)"]
)

# Align indices
common_idx = ts_result.feature_matrix.index.intersection(target_per_entity.index)
target_aligned = target_per_entity.loc[common_idx]

ts_filtered = ts_engine.filter_relevant_features(
    result=ts_result,
    target=target_aligned,
    fdr_level=0.05
)

print(f"Relevant features: {len(ts_filtered.relevant_features or [])} / {len(ts_result.feature_names)}")
if ts_filtered.relevant_features:
    for name in ts_filtered.relevant_features[:10]:
        print(f"  - {name}")

# Merge tsfresh features back into the main DataFrame (entity-level -> transaction-level join)
# Use filtered features if available, otherwise fall back to all extracted features
ts_to_merge = ts_filtered.relevant_features if ts_filtered.relevant_features else ts_result
# ts_to_merge
# df = ts_engine.to_spark(ts_to_merge, df, id_column=index_col)

ts_filtered = ts_filtered.feature_matrix[ts_to_merge]
ts_filtered = ts_filtered.reset_index()
ts_filtered = ts_filtered.rename(columns={'index': index_col})

df = pandas_to_spark(ts_filtered, spark)

#Drop original columns
# df = df.drop(*col_to_drop)

print(f"\nDataFrame after tsfresh merge: {len(df.columns)} columns")

In [0]:
df.count(), df.select('customer_id').distinct().count()


## Feature Table

Saving data in feature tables

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient

fe = FeatureEngineeringClient()

# Create feature table with `trans_num` as the primary key.
# Take schema from DataFrame output by featuretool_features
customer_feature_table = fe.create_table(
  name=tsfresh_feature_table_name,
  # primary_keys=[index_col, time_index],
  # timeseries_columns=time_index,
  primary_keys = [index_col],
  schema=df.schema,
  description='Time Series features'
)


In [0]:
fe.write_table(
  name=tsfresh_feature_table_name,
  df = df,
  mode = 'merge'
)

## TODO: Add Support for categorical features